# Computational Derivations and Spectral Invariants of Girard Torsion, Jordan Algebra H3(O), and 3-Torus Eigenmodes Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We cover data inspection, extraction, cleaning, and visualization, referencing entities by their `@id` fields as per Croissant specification.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [`https://sen.science/doi/10.71728/senscience.k1n2-dzgt/fair2.json`](https://sen.science/doi/10.71728/senscience.k1n2-dzgt/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and preview its description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.k1n2-dzgt/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out metadata overview
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.
We enumerate all record sets, fields, and columns in the dataset, referencing entities by their `@id`.

In [ ]:
# Discover record sets in the dataset
record_sets = list(dataset.record_sets)
print(f"Record Sets found ({len(record_sets)}):")
for record_set in record_sets:
    print(f"- @id: {record_set['@id']} | name: {record_set.get('name', '')}")

# Display fields and columns in each record set
for record_set in record_sets:
    print(f"\nRecord Set: {record_set['@id']} ({record_set.get('name', '')})")
    fields = record_set.get('field', [])
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    Field @id: {field['@id']} | name: {field.get('name', '')}")
            columns = field.get('column', [])
            if columns:
                print("      Columns:")
                for column in columns:
                    print(f"        Column @id: {column['@id']} | name: {column.get('name', '')}")
    else:
        print("  No fields defined.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.
All entities are referenced by their `@id` fields.

In [ ]:
# Prepare DataFrames for each record set by @id
dfs = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        dfs[rs_id] = pd.DataFrame(records)
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

# Show columns for the first record set if available
if dfs:
    first_rs_id = record_set_ids[0]
    print(f"Columns for record set {first_rs_id}: {dfs[first_rs_id].columns.tolist()}")
    dfs[first_rs_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Processing: Filter records, normalize numeric fields, and group by key attributes.
We reference fields and columns strictly by their `@id`.

In [ ]:
# Example EDA: Choose the first available record set and identify numeric fields by @id
import numpy as np

# Select record set
if dfs:
    rs_id = list(dfs.keys())[0]
    df = dfs[rs_id]
    print(f"Using record set: {rs_id}")

    # Identify numeric columns (@id) by inferring dtype
    numeric_cols = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    print(f"Numeric columns (@id): {numeric_cols}")

    # If unavailable, demonstrate with placeholder
    if not numeric_cols:
        print("No numeric columns found. Simulating numeric EDA with a dummy numeric column if present.")
        if 'dummy_numeric' in df.columns:
            numeric_field_id = 'dummy_numeric'
        else:
            df['dummy_numeric'] = np.random.uniform(1, 100, len(df))
            numeric_field_id = 'dummy_numeric'
    else:
        numeric_field_id = numeric_cols[0]

    # Filtering
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    colnorm = f"{numeric_field_id}_normalized"
    filtered_df[colnorm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, colnorm]].head())

    # Grouping: pick a field to group by (if categorical present)
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No categorical grouping field found.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields referenced by their `@id`.
Let's plot the normalized numeric column distribution for the first record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize if available
if dfs:
    rs_id = list(dfs.keys())[0]
    df = dfs[rs_id]
    numeric_cols = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if not numeric_cols:
        numeric_field_id = 'dummy_numeric'
    else:
        numeric_field_id = numeric_cols[0]
    colnorm = f"{numeric_field_id}_normalized"

    # Make plot if normalized column exists from EDA
    if colnorm in df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[colnorm].dropna(), kde=True)
        plt.title(f"Normalized distribution of {numeric_field_id} (@id: {numeric_field_id})")
        plt.xlabel(colnorm)
        plt.ylabel('Frequency')
        plt.show()
    else:
        if numeric_field_id in df.columns:
            plt.figure(figsize=(8, 5))
            sns.histplot(df[numeric_field_id].dropna(), kde=True)
            plt.title(f"Distribution of {numeric_field_id} (@id: {numeric_field_id})")
            plt.xlabel(numeric_field_id)
            plt.ylabel('Frequency')
            plt.show()
        else:
            print("No numeric or normalized column found for plotting.")
else:
    print("No loaded DataFrames available for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using `mlcroissant`, adhering strictly to referencing all entities by `@id`. We loaded metadata, enumerated record sets, fields, and columns, extracted data, applied basic filtering and normalization, and visualized key numeric fields. This approach enables reproducible, FAIR-compliant scientific workflows and can be extended to deeper analyses or model-building depending on the dataset structure and use case.